In [1]:
import os, shutil, subprocess, json, sys
from pathlib import Path

DOCKER_BIN = os.path.join(os.environ.get('ProgramFiles', 'C:/Program Files'),
                          'Docker', 'Docker', 'resources', 'bin')
if os.path.isdir(DOCKER_BIN) and DOCKER_BIN not in os.environ.get('PATH', ''):
    os.environ['PATH'] = DOCKER_BIN + os.pathsep + os.environ.get('PATH', '')
    print(f'  (PATH ditambah: {DOCKER_BIN})')

def jalankan(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return r.returncode, (r.stdout or "") + (r.stderr or "")
    except Exception as e:
        return -1, str(e)

print("PRASYARAT")
print("=" * 62)

import psutil
vm = psutil.virtual_memory()
du = shutil.disk_usage("D:\\")
duc = shutil.disk_usage("C:\\")
print(f"  RAM total          : {vm.total/1e9:,.1f} GB")
print(f"  CPU logis          : {os.cpu_count()}")
print(f"  D: bebas           : {du.free/1e9:,.1f} GB")
print(f"  C: bebas           : {duc.free/1e9:,.1f} GB")

rc, out = jalankan('powershell -NoProfile -Command "(Get-CimInstance Win32_ComputerSystem).HypervisorPresent"')
print(f"  Virtualisasi aktif : {out.strip()}")

rc, out = jalankan("wsl --status")
print(f"  WSL terpasang      : {'YA' if rc == 0 else 'BELUM'}")
rc, out = jalankan("wsl --list --quiet")
distro = [d.strip() for d in out.replace('\x00', '').splitlines() if d.strip()]
print(f"  Distro WSL         : {distro if distro else 'BELUM ADA'}")

rc, out = jalankan("docker --version")
punya_docker = rc == 0
print(f"  Docker             : {out.strip() if punya_docker else 'BELUM TERPASANG'}")

rc, out = jalankan("docker compose version")
print(f"  Docker Compose     : {out.strip() if rc == 0 else '-'}")

svc = jalankan('powershell -NoProfile -Command "(Get-Service MySQL80).Status"')[1].strip()
print(f"  MySQL80            : {svc or 'tidak ditemukan'}")

WSLCONFIG = Path(os.environ["USERPROFILE"]) / ".wslconfig"
print(f"  .wslconfig         : {'ADA' if WSLCONFIG.exists() else 'BELUM ADA'}")

PRASYARAT
  RAM total          : 68.4 GB
  CPU logis          : 28
  D: bebas           : 514.3 GB
  C: bebas           : 69.1 GB
  Virtualisasi aktif : True
  WSL terpasang      : YA
  Distro WSL         : ['Ubuntu', 'docker-desktop']
  Docker             : Docker version 29.7.2, build a7dcaa6
  Docker Compose     : Docker Compose version v5.4.0
  MySQL80            : Running
  .wslconfig         : ADA


In [2]:
sumber = Path(r"D:\BDA\docker\wslconfig.txt")
isi = sumber.read_text(encoding="utf-8")
print(isi)
print("=" * 62)

if WSLCONFIG.exists():
    print(f"SUDAH ADA di {WSLCONFIG} -- isi lama TIDAK ditimpa.")
    print("Bandingkan sendiri dengan isi di atas bila perlu:\n")
    print(WSLCONFIG.read_text(encoding="utf-8"))

[wsl2]
memory=48GB
processors=24
swap=16GB
localhostForwarding=true

autoMemoryReclaim=gradual
sparseVhd=true

SUDAH ADA di C:\Users\Willy Boen\.wslconfig -- isi lama TIDAK ditimpa.
Bandingkan sendiri dengan isi di atas bila perlu:

﻿[wsl2]
memory=48GB
processors=24
swap=16GB
localhostForwarding=true

[experimental]
autoMemoryReclaim=gradual
sparseVhd=true



In [3]:
# Jalankan ulang sel ini setelah Langkah 2 dan 3 selesai.
rc, out = jalankan("docker --version")
if rc != 0:
    print("Docker belum terpasang. Selesaikan Langkah 2 dulu.")
else:
    print(out.strip())
    rc, out = jalankan("docker info --format \"{{.DockerRootDir}} | {{.MemTotal}} | {{.NCPU}}\"")
    print("  root dir | memori | cpu :", out.strip())
    rc, out = jalankan("docker info --format \"{{.OperatingSystem}}\"")
    print("  backend :", out.strip())
    if "D:" not in jalankan("docker info")[1] and "docker-desktop" in out.lower():
        print("\n  Catatan: pastikan Disk image location sudah menunjuk ke D:\\docker.")

Docker version 29.7.2, build a7dcaa6
  root dir | memori | cpu : /var/lib/docker | 50511613952 | 24
  backend : Docker Desktop


In [4]:
DOCKER_DIR = r"D:\BDA\docker"

# Klaster punya DUA profil, dan hanya satu yang boleh hidup pada satu waktu:
#
#   yarn        HDFS + YARN (ResourceManager + 3 NodeManager) + jupyter
#   standalone  HDFS + Spark Standalone (master + 3 worker) + jupyter
#
# RAM WSL 48 GB, sedangkan pekerja masing-masing profil meminta sekitar
# 25 GB, sehingga keduanya tidak muat bersamaan. Notebook 03 menuntut
# profil yarn; notebook 04 sampai 07 bisa berjalan di keduanya.
PROFIL = "yarn"

if jalankan("docker --version")[0] != 0:
    print("Docker belum ada -- lewati sel ini.")
else:
    print("Membangun image jupyter (beberapa menit pada kali pertama) ...")
    print("Image ini memuat Spark, PySpark, dan klien Hadoop lengkap, sehingga")
    print("container jupyter berperan sebagai gateway node: tempat job disubmit.")
    rc, out = jalankan(f'cd /d "{DOCKER_DIR}" && docker compose build jupyter',
                       timeout=3600)
    print(out[-2000:])

    # Profil lawan dimatikan dulu supaya container lama tidak ikut
    # memperebutkan RAM dengan yang baru dinyalakan.
    lain = "standalone" if PROFIL == "yarn" else "yarn"
    print(f"\nMematikan profil {lain} bila masih hidup ...")
    jalankan(f'cd /d "{DOCKER_DIR}" && docker compose --profile {lain} down',
             timeout=300)

    print(f"\nMenyalakan klaster dengan profil {PROFIL} ...")
    # BDA_SPARK_MASTER dibaca docker-compose lalu diteruskan ke container
    # jupyter, dan dari sana dibaca bda_common untuk menentukan penjadwal.
    # Tanda kutip mengapit SELURUH penugasan. Tanpa itu, cmd.exe ikut
    # menyimpan spasi sebelum && ke dalam nilainya -- "yarn " alih-alih
    # "yarn" -- dan Spark menolaknya dengan galat yang tidak menyebut
    # spasi sama sekali.
    awalan = 'set "BDA_SPARK_MASTER=yarn" && ' if PROFIL == "yarn" else ""
    rc, out = jalankan(
        f'cd /d "{DOCKER_DIR}" && {awalan}docker compose --profile {PROFIL} up -d',
        timeout=1200)
    print(out[-2000:])

    rc, out = jalankan(f'cd /d "{DOCKER_DIR}" && docker compose --profile {PROFIL} ps')
    print(out)

Membangun image jupyter (beberapa menit pada kali pertama) ...
Image ini memuat Spark, PySpark, dan klien Hadoop lengkap, sehingga
container jupyter berperan sebagai gateway node: tempat job disubmit.
on/hadoop-${HADOOP_VERSION}/hadoop-${HADOOP_VERSION}.tar.gz\"  && mkdir -p /opt/hadoop  && tar -xzf /tmp/hadoop.tgz -C /opt/hadoop --strip-components=1  && rm -f /tmp/hadoop.tgz  && rm -rf /opt/hadoop/share/doc /opt/hadoop/share/hadoop/*/sources  && mkdir -p /opt/hadoop-conf" did not complete successfully: exit code: 56
------
 > [7/8] RUN curl -fsSL -o /tmp/hadoop.tgz       "https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz"  && mkdir -p /opt/hadoop  && tar -xzf /tmp/hadoop.tgz -C /opt/hadoop --strip-components=1  && rm -f /tmp/hadoop.tgz  && rm -rf /opt/hadoop/share/doc /opt/hadoop/share/hadoop/*/sources  && mkdir -p /opt/hadoop-conf:
7011.3 curl: (56) OpenSSL SSL_read: error:0A000126:SSL routines::unexpected eof while reading, errno 0
------
 Image bda-jupy

In [6]:
import time

def dexec(svc, perintah, timeout=1800):
    return jalankan(f'docker exec bda-{svc} bash -lc "{perintah}"', timeout=timeout)

if jalankan("docker --version")[0] != 0:
    print("Docker belum ada -- lewati sel ini.")
else:
    print("Menunggu namenode keluar dari safe mode ...")
    for i in range(30):
        rc, out = dexec("namenode", "hdfs dfsadmin -safemode get", timeout=90)
        if "OFF" in out:
            print("  safe mode OFF")
            break
        time.sleep(10)
    else:
        print("  masih ON -- paksa keluar")
        dexec("namenode", "hdfs dfsadmin -safemode leave")

    print("\nMembuat struktur direktori HDFS ...")
    zona = ["lake/fasta", "lake/meta_csv", "stage", "features",
            "models", "graph", "mart", "spark-events"]
    dexec("namenode", "hdfs dfs -mkdir -p " + " ".join(f"/bda/{z}" for z in zona))
    dexec("namenode", "hdfs dfs -chmod -R 777 /bda")

    print("Menyalin zona bronze dari Windows ke HDFS ...")
    # Container jupyter melihat D:\BDA sebagai /workspace, jadi salin dari sana.
    rc, out = dexec("jupyter",
                    "export HADOOP_CONF_DIR=/dev/null; "
                    "/opt/spark/bin/spark-submit --version 2>/dev/null; "
                    "echo ok", timeout=120)
    rc, out = jalankan(
        'docker exec bda-namenode bash -lc '
        '"hdfs dfs -ls /bda && hdfs dfs -df -h /"', timeout=120)
    print(out)

Menunggu namenode keluar dari safe mode ...
  safe mode OFF

Membuat struktur direktori HDFS ...
Menyalin zona bronze dari Windows ke HDFS ...
Found 8 items
drwxrwxrwx   - root supergroup          0 2026-09-22 02:50 /bda/features
drwxrwxrwx   - root supergroup          0 2026-09-02 13:02 /bda/graph
drwxrwxrwx   - root supergroup          0 2026-09-02 13:02 /bda/lake
drwxrwxrwx   - root supergroup          0 2026-09-02 13:02 /bda/mart
drwxrwxrwx   - root supergroup          0 2026-09-22 07:27 /bda/models
drwxrwxrwx   - root supergroup          0 2026-09-22 02:21 /bda/mr
drwxrwxrwx   - root supergroup          0 2026-09-22 08:10 /bda/spark-events
drwxrwxrwx   - root supergroup          0 2026-09-22 02:13 /bda/stage
Filesystem             Size    Used  Available  Use%
hdfs://namenode:8020  2.9 T  88.8 G      2.5 T    3%



In [7]:
if jalankan("docker --version")[0] == 0:
    print("Menyalin berkas lake ke HDFS (bisa lama bila arsip penuh sudah diunduh) ...")
    rc, out = jalankan(
        'docker run --rm --network bda-influenza_bda '
        '-v "D:/BDA:/src" apache/hadoop:3.3.6 bash -lc '
        '"hdfs dfs -fs hdfs://namenode:8020 -put -f /src/lake/fasta/*.fasta.gz /bda/lake/fasta/ ; '
        'hdfs dfs -fs hdfs://namenode:8020 -put -f /src/lake/meta_csv/*.csv /bda/lake/meta_csv/ ; '
        'hdfs dfs -fs hdfs://namenode:8020 -du -s -h /bda/lake"',
        timeout=7200)
    print(out[-3000:])

Menyalin berkas lake ke HDFS (bisa lama bila arsip penuh sudah diunduh) ...
529.8 M  1.6 G  /bda/lake



In [8]:
if jalankan("docker --version")[0] == 0:
    import urllib.request

    print("=== LAPORAN HDFS ===")
    print(dexec("namenode", "hdfs dfsadmin -report | head -22")[1])

    print("=== LAPORAN YARN ===")
    # Dibaca dari REST API ResourceManager. Kalau bagian ini kosong,
    # berarti tidak ada manajer sumber daya yang menerima job -- dan
    # MapReduce akan diam-diam jatuh ke mode "local" tanpa memberi tahu.
    try:
        with urllib.request.urlopen(
                "http://localhost:8088/ws/v1/cluster/metrics", timeout=8) as r:
            d = json.load(r)["clusterMetrics"]
        print(f"  NodeManager aktif : {d['activeNodes']}")
        print(f"  Memori klaster    : {d['totalMB']:,} MB")
        print(f"  vCore klaster     : {d['totalVirtualCores']}")
        print(f"  Aplikasi selesai  : {d['appsCompleted']}  "
              f"gagal: {d['appsFailed']}")
    except Exception as e:
        print(f"  ResourceManager belum siap ({type(e).__name__}).")
        print("  Klaster dinyalakan dengan profil yarn?")

    print()
    for nama, url in [("HDFS namenode", "http://localhost:9870"),
                      ("YARN ResourceManager", "http://localhost:8088"),
                      ("Spark master", "http://localhost:8080"),
                      ("JupyterLab", "http://localhost:8888")]:
        try:
            kode = urllib.request.urlopen(url, timeout=8).status
            print(f"  {nama:<22} {url:<26} HTTP {kode}")
        except Exception as e:
            print(f"  {nama:<22} {url:<26} belum siap ({type(e).__name__})")

    print()
    print("  Catatan: pada profil yarn, Spark master memang TIDAK hidup --")
    print("  Spark memakai YARN sebagai penjadwalnya. Sebaliknya pada profil")
    print("  standalone, ResourceManager yang tidak hidup. Satu baris 'belum")
    print("  siap' di antara keduanya adalah hal yang diharapkan.")

=== LAPORAN HDFS ===
Configured Capacity: 3243303530496 (2.95 TB)
Present Capacity: 2860717723648 (2.60 TB)
DFS Remaining: 2765354151936 (2.52 TB)
DFS Used: 95363571712 (88.81 GB)
DFS Used%: 3.33%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (3):


=== LAPORAN YARN ===
  NodeManager aktif : 3
  Memori klaster    : 18,432 MB
  vCore klaster     : 12
  Aplikasi selesai  : 0  gagal: 0

  HDFS namenode          http://localhost:9870      HTTP 200
  YARN ResourceManager   http://localhost:8088      HTTP 200
  Spark master     

In [16]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymysql"], check=False)
sys.path.insert(0, r"D:\BDA\nb")
from bda_common import SECRET_PATH, muat_secret

try:
    rahasia = muat_secret()
except FileNotFoundError as e:
    print(e)
    rahasia = None

if not rahasia or "ISI_PASSWORD" in rahasia["mysql"]["password"]:
    print(f"  Isi dulu kredensial MySQL di {SECRET_PATH}, lalu jalankan ulang sel ini.")
else:
    import pymysql
    m = rahasia["mysql"]
    print(f"  Menghubungi {m['user']}@{m['host']}:{m['port']}/{m['database']}")
    try:
        kon = pymysql.connect(host=m["host"], port=m["port"], user=m["user"],
                              password=m["password"], database=m["database"],
                              connect_timeout=15)
        with kon.cursor() as c:
            c.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
            versi, db, akun = c.fetchone()
            print(f"  MySQL versi     : {versi}")
            print(f"  Database aktif  : {db}")
            print(f"  Terhubung as    : {akun}")

            # SHOW GRANTS berlaku untuk akun yang sedang login, jadi tidak
            # memerlukan hak baca ke tabel sistem. Ini penting: akun proyek
            # sengaja dibatasi hanya pada database bda_influenza, sehingga
            # kueri ke mysql.user memang DITOLAK -- dan penolakan itu justru
            # bukti bahwa prinsip hak akses minimum sudah diterapkan, bahan
            # yang bisa dikutip untuk Bab 3.1 Security.
            c.execute("SHOW GRANTS FOR CURRENT_USER()")
            print("  Hak akses akun  :")
            for (baris,) in c.fetchall():
                print(f"    {baris}")

            c.execute("SELECT COUNT(*) FROM information_schema.TABLES "
                      "WHERE TABLE_SCHEMA = %s", (m["database"],))
            print(f"  Tabel saat ini  : {c.fetchone()[0]} "
                  f"(akan diisi notebook 07)")
        kon.close()
        print("\n  SIAP. Notebook 07 dapat menulis tabel mart ke MySQL.")
    except Exception as e:
        print(f"  Gagal terhubung: {e}")
        print("\n  Bila ditolak, jalankan reset dari Terminal (Admin):")
        print("    powershell -ExecutionPolicy Bypass -File D:\\BDA\\docker\\reset-mysql.ps1")

  Menghubungi bda@127.0.0.1:3306/bda_influenza
  MySQL versi     : 8.0.46
  Database aktif  : bda_influenza
  Terhubung as    : bda@%
  Hak akses akun  :
    GRANT USAGE ON *.* TO `bda`@`%`
    GRANT ALL PRIVILEGES ON `bda_influenza`.* TO `bda`@`%`
  Tabel saat ini  : 0 (akan diisi notebook 07)

  SIAP. Notebook 07 dapat menulis tabel mart ke MySQL.


**http://localhost:8888** (token `bda`) -> buka folder `nb` -> jalankan `.ipynb`.

Di dalam klaster, `bda_common` otomatis beralih ke mode klaster:
`jalur("stage/sequences")` menunjuk ke `hdfs://namenode:8020/bda/stage/sequences`,
dan Spark terhubung ke penjadwal sesuai profil yang aktif. Tidak ada satu sel pun
yang perlu diubah.

# Dua profil klaster

| | `yarn` | `standalone` |
|---|---|---|
| Penjadwal | YARN ResourceManager | Spark Standalone master |
| Pekerja | 3 NodeManager | 3 Spark worker |
| Image pekerja | `apache/hadoop:3.3.6` | `bda-jupyter:3.5.9` |
| Python di pekerja | **tidak ada** | Python 3.10 + numpy |
| Antarmuka | localhost:8088 | localhost:8080 |
| Notebook | **03** | **04, 05, 06** |
| Bebas dipakai | 01, 02, 07 | 01, 02, 07 |

Keduanya **tidak bisa hidup bersamaan**: RAM WSL 48 GB, sedangkan pekerja
masing-masing profil meminta sekitar 25 GB.

### Kenapa sebagian notebook terikat pada satu profil

Yang menentukan bukan selera, melainkan **ada tidaknya Python di executor**.

NodeManager YARN memakai image `apache/hadoop:3.3.6` yang berbasis CentOS 7
dan tidak membawa Python sama sekali. Selama seluruh transformasi ditulis
sebagai Spark SQL asli, itu tidak jadi masalah: executor cukup menjalankan JVM.
Notebook 01, 02, 03, dan 07 seluruhnya seperti itu.

Begitu ada satu saja `F.udf(...)` atau sentuhan ke `.rdd`, Spark harus
menyalakan proses Python di setiap executor. Di NodeManager proses itu tidak
ada, dan job gagal dengan:

```
java.io.IOException: error=2, No such file or directory
    at org.apache.spark.api.python.PythonWorkerFactory.create
```

Pesan itu tidak menyebut Python sama sekali, jadi mudah disalahartikan sebagai
masalah berkas atau izin. Tiga notebook yang terkena:

| Notebook | Baris pemicunya |
|---|---|
| 04 | `F.udf(lambda v: int(v.numNonzeros()), "int")` |
| 05 | `F.udf(lambda v: float(max(v)), "double")` (dua tempat) |
| 06 | `t.rdd.isEmpty()` |

Pekerja profil `standalone` memakai image `bda-jupyter` yang memuat Python 3.10
beserta numpy, sehingga ketiganya berjalan normal di sana.

# Perintah harian

```powershell
cd D:\BDA\docker

# menyalakan -- pilih salah satu
set BDA_SPARK_MASTER=yarn
docker compose --profile yarn up -d
docker compose --profile standalone up -d

# berpindah profil: matikan yang lama dulu
docker compose --profile standalone down
docker compose --profile yarn up -d

# memeriksa
docker compose --profile yarn ps
docker compose logs -f jupyter

# menghentikan
docker compose --profile yarn stop
docker compose --profile yarn down
docker compose --profile yarn down -v     # HATI-HATI: menghapus volume HDFS
```

Perintah terakhir menghapus seluruh isi HDFS, termasuk feature store yang
memakan berjam-jam untuk dibangun. Container boleh dibuang kapan saja; volume
`bda-influenza_nn` dan `dn1..3` tidak.